# SAM2-UNet 4-channel (RGB + DTM) — corrected debugging workflow

This notebook replaces the submitted notebook's mixed environments, hard-coded paths, stale outputs, and legacy experiments with a restart-safe workflow.

**Measured evidence extracted from the submitted run**

- 910 training tiles and 184 validation tiles; each split had 50% positive tiles.
- Foreground pixels differed substantially: train `0.021482` versus validation `0.006562` (**3.27× shift**).
- Best validation IoU and best validation loss both occurred at **epoch 1**: IoU `0.074921`, loss `4.164995`.
- By epoch 29, train IoU reached `0.902072`, while validation IoU was `0.000000`; validation loss rose to `10.022160`.
- The final generalisation gap was `0.902072`; validation loss was **11.38×** training loss.

This pattern is severe overfitting and/or data-domain shift. It is not evidence that the fourth-channel convolution itself is malformed. The corrected pipeline therefore audits pairing, masks, DTM nodata/normalisation, split leakage, and threshold behaviour before training.

> Full SAM2 execution still requires your `sam2_hiera_large.pt` checkpoint and RGB/mask/DTM folders. They were not included in the uploaded ZIP, so the delivered static and synthetic tests cannot substitute for a full data-level run.


In [ ]:
# Runtime identity: every subprocess in this notebook uses this exact interpreter.
import os
import platform
import subprocess
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version   :", sys.version)
print("Platform         :", platform.platform())

try:
    import torch
    import torchvision
    print("torch            :", torch.__version__)
    print("torchvision      :", torchvision.__version__)
    print("CUDA available   :", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU              :", torch.cuda.get_device_name(0))
except Exception as exc:
    print("PyTorch check failed:", repr(exc))


## 1. Single configuration cell

Edit only this cell. The model checkpoint and dataset are intentionally not guessed.


In [ ]:
# ----------------------------- EDIT THESE PATHS -----------------------------
# Colab example: upload/extract the delivered SAM2-UNet-debugged.zip to /content,
# or point PROJECT_DIR to its location in Google Drive.
PROJECT_DIR = Path("/content/SAM2-UNet-debugged")
HIERA_PATH = Path("/content/sam2_hiera_large.pt")

TRAIN_IMAGE = Path("/content/data/train/IMAGE")
TRAIN_MASK  = Path("/content/data/train/LABEL")
TRAIN_DTM   = Path("/content/data/train/DTM_NORM")

VAL_IMAGE = Path("/content/data/val/IMAGE")
VAL_MASK  = Path("/content/data/val/LABEL")
VAL_DTM   = Path("/content/data/val/DTM_NORM")

TEST_IMAGE = Path("/content/data/test/IMAGE")
TEST_MASK  = Path("/content/data/test/LABEL")
TEST_DTM   = Path("/content/data/test/DTM_NORM")

RUN_DIR = Path("/content/runs/sam2_unet_rgb_dtm_100pct")
PRED_DIR = Path("/content/predictions/sam2_unet_rgb_dtm_100pct")

# Reproducible experiment controls
MODEL_CFG = "sam2_hiera_l.yaml"
IMAGE_SIZE = 512
DTM_NORM = "per_tile_zscore"   # also test uint16_01 as an explicit ablation
DTM_SCALE = 1.0
DTM_AVAILABILITY = 1.0          # 100% DTM run; do not change to 0.0 while labelling it "100%"
AVAILABILITY_SEED = 42
MASK_THRESHOLD = 0.0            # change to 127 only after auditing mask values
POS_WEIGHT = 12.0              # treat as a tunable ablation, not a universal constant
SEED = 42

# Safety toggles: long/heavy operations never start accidentally.
INSTALL_CORE_DEPENDENCIES = True
RUN_FULL_MODEL_CONSISTENCY = False
RUN_TRAINING = False
RUN_TESTING = False
RUN_EVALUATION = False
# ---------------------------------------------------------------------------


## 2. Optional Colab Drive mount and dependency installation

Do not create/activate a conda environment or use `!source .../activate` inside notebook cells. Shell activation does not persist into the next `!` call. `%pip`/`sys.executable -m pip` installs into the active kernel.


In [ ]:
# Optional Google Drive mount; harmlessly skipped outside Colab.
try:
    from google.colab import drive
    # Uncomment only when PROJECT_DIR/data live in Drive.
    # drive.mount('/content/drive')
    print("Colab detected. Drive mount remains opt-in.")
except ImportError:
    print("Not running in Colab.")

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"PROJECT_DIR does not exist: {PROJECT_DIR}\n"
        "Extract the delivered SAM2-UNet-debugged.zip or edit the configuration cell."
    )

if INSTALL_CORE_DEPENDENCIES:
    requirements = PROJECT_DIR / "requirements-core.txt"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", str(requirements)])

# Ensure imports resolve to the corrected project.
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))
print("Working directory:", Path.cwd())


## 3. Static and synthetic regression tests

Expected result: `20 passed`. These tests do not need the SAM2 checkpoint or real dataset.


In [ ]:
subprocess.check_call([sys.executable, "-m", "compileall", "-q", str(PROJECT_DIR)])
subprocess.check_call([sys.executable, "-m", "pytest", "-q", str(PROJECT_DIR / "tests")])
print("Static compilation and synthetic tests passed.")


## 4. Validate all external paths before reading data


In [ ]:
required_files = {}
required_dirs = {
    "TRAIN_IMAGE": TRAIN_IMAGE, "TRAIN_MASK": TRAIN_MASK, "TRAIN_DTM": TRAIN_DTM,
    "VAL_IMAGE": VAL_IMAGE, "VAL_MASK": VAL_MASK, "VAL_DTM": VAL_DTM,
}

if RUN_FULL_MODEL_CONSISTENCY or RUN_TRAINING or RUN_TESTING:
    required_files["HIERA_PATH"] = HIERA_PATH
if RUN_TESTING:
    required_dirs.update({
        "TEST_IMAGE": TEST_IMAGE, "TEST_MASK": TEST_MASK, "TEST_DTM": TEST_DTM,
    })
if RUN_EVALUATION:
    required_dirs["TEST_MASK"] = TEST_MASK

missing = []
for name, path in required_files.items():
    if not path.is_file():
        missing.append(f"{name}: missing file {path}")
for name, path in required_dirs.items():
    if not path.is_dir():
        missing.append(f"{name}: missing directory {path}")

if missing:
    raise FileNotFoundError("Fix the configuration cell:\n- " + "\n- ".join(missing))
print("All paths required by the enabled stages exist.")


## 5. Dataset audit — run before model construction

This audit is strict and case-insensitive. It rejects duplicate stems and missing masks/DTMs instead of silently dropping samples. It also quantifies mask threshold sensitivity, DTM nodata, data types, dimensions, and constant tiles.


In [ ]:
AUDIT_DIR = RUN_DIR / "audits"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

def run_audit(split, image_path, mask_path, dtm_path):
    output = AUDIT_DIR / f"{split}_audit.json"
    command = [
        sys.executable, str(PROJECT_DIR / "audit_dataset.py"),
        "--image_path", str(image_path),
        "--mask_path", str(mask_path),
        "--dtm_path", str(dtm_path),
        "--output_json", str(output),
        "--mask_threshold", str(MASK_THRESHOLD),
    ]
    subprocess.check_call(command)
    return output

train_audit_path = run_audit("train", TRAIN_IMAGE, TRAIN_MASK, TRAIN_DTM)
val_audit_path = run_audit("val", VAL_IMAGE, VAL_MASK, VAL_DTM)
print(train_audit_path)
print(val_audit_path)


In [ ]:
import json

def load_summary(path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)["summary"]

train_summary = load_summary(train_audit_path)
val_summary = load_summary(val_audit_path)

train_fg = train_summary["foreground_ratio_at_config_threshold"]
val_fg = val_summary["foreground_ratio_at_config_threshold"]
shift = max(train_fg, val_fg) / max(min(train_fg, val_fg), 1e-12)

print("Train matched samples:", train_summary["pairing"]["matched_count"])
print("Val matched samples  :", val_summary["pairing"]["matched_count"])
print("Train foreground ratio:", train_fg)
print("Val foreground ratio  :", val_fg)
print("Foreground-ratio shift:", shift, "x")
train_values = list(train_summary["mask_unique_values"].keys())
val_values = list(val_summary["mask_unique_values"].keys())
print("Train mask unique values:", len(train_values), "preview:", train_values[:32])
print("Val mask unique values  :", len(val_values), "preview:", val_values[:32])

if shift >= 2:
    print("WARNING: >2x foreground-distribution shift. Verify split construction/domain balance.")
if train_summary["shape_mismatch_tiles"] or val_summary["shape_mismatch_tiles"]:
    raise RuntimeError("Shape mismatches exist; inspect audit CSV files before training.")


### Manual co-registration check

Equal array sizes do **not** prove RGB and DTM are spatially aligned. PNG/JPEG RGB tiles have no CRS/affine transform. Inspect representative pairs and verify the upstream tiling transform.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from dataset import build_sample_records, read_dtm_masked

records, _ = build_sample_records(TRAIN_IMAGE, TRAIN_DTM, TRAIN_MASK, strict=True)
record = records[0]
rgb = np.asarray(Image.open(record.image_path).convert("RGB"))
mask = np.asarray(Image.open(record.mask_path).convert("L"))
dtm, dtm_meta = read_dtm_masked(record.dtm_path)

print("Stem:", record.stem)
print("RGB / mask / DTM shapes:", rgb.shape, mask.shape, dtm.shape)
print("DTM metadata:", dtm_meta)

plt.figure(figsize=(7, 7)); plt.imshow(rgb); plt.title(f"RGB — {record.stem}"); plt.axis("off"); plt.show()
plt.figure(figsize=(7, 7)); plt.imshow(dtm); plt.title("DTM (raw finite values)"); plt.axis("off"); plt.show()
plt.figure(figsize=(7, 7)); plt.imshow(mask, cmap="gray"); plt.title("Mask"); plt.axis("off"); plt.show()


## 6. Full SAM2 zero-init consistency check

This requires the pretrained SAM2 checkpoint and enough RAM/VRAM. At initialisation, outputs for random DTM and zero DTM must match because the new DTM weight slice is exactly zero.


In [ ]:
if RUN_FULL_MODEL_CONSISTENCY:
    subprocess.check_call([
        sys.executable, str(PROJECT_DIR / "verify_consistency.py"),
        "--hiera_path", str(HIERA_PATH),
        "--model_cfg", MODEL_CFG,
        "--size", "256",
        "--output_json", str(AUDIT_DIR / "model_consistency.json"),
    ])
else:
    print("Skipped. Set RUN_FULL_MODEL_CONSISTENCY=True after path/data audit passes.")


## 7. Train

Important corrections relative to the submitted run:

- `--epochs` is the total target epoch, including a resumed checkpoint.
- Early stopping defaults to 8; the submitted log would have stopped at epoch 9 instead of continuing to epoch 29.
- The checkpoint records normalisation, DTM availability, seed, mask threshold, split hashes, validation threshold sweep, and class metrics.
- Existing logs are never silently appended unless `--resume` is used.


In [ ]:
TRAIN_COMMAND = [
    sys.executable, str(PROJECT_DIR / "train.py"),
    "--hiera_path", str(HIERA_PATH),
    "--model_cfg", MODEL_CFG,
    "--train_image_path", str(TRAIN_IMAGE),
    "--train_mask_path", str(TRAIN_MASK),
    "--train_dtm_path", str(TRAIN_DTM),
    "--val_image_path", str(VAL_IMAGE),
    "--val_mask_path", str(VAL_MASK),
    "--val_dtm_path", str(VAL_DTM),
    "--save_path", str(RUN_DIR),
    "--trainsize", str(IMAGE_SIZE),
    "--dtm_norm", DTM_NORM,
    "--dtm_scale", str(DTM_SCALE),
    "--dtm_availability", str(DTM_AVAILABILITY),
    "--availability_seed", str(AVAILABILITY_SEED),
    "--mask_threshold", str(MASK_THRESHOLD),
    "--epochs", "50",
    "--batch_size", "4",
    "--lr", "0.0001",
    "--pos_weight", str(POS_WEIGHT),
    "--patience", "8",
    "--seed", str(SEED),
]
print(" ".join(map(str, TRAIN_COMMAND)))

if RUN_TRAINING:
    subprocess.check_call(TRAIN_COMMAND)
else:
    print("Training skipped. Set RUN_TRAINING=True only after all audits pass.")


## 8. Analyse the training log

`train_iou` is an online training-mode metric on augmented batches and is not perfectly comparable to evaluation-mode validation IoU. Use it as an optimisation signal, not as a generalisation estimate.


In [ ]:
import pandas as pd

log_path = RUN_DIR / "log.csv"
if not log_path.exists() and (RUN_DIR / "log_v2.csv").exists():
    log_path = RUN_DIR / "log_v2.csv"

if log_path.exists():
    log = pd.read_csv(log_path)
    display(log.tail())
    best_row = log.loc[log["val_iou"].idxmax()]
    print("Best fixed-threshold validation IoU:")
    display(best_row.to_frame("value"))

    plt.figure(figsize=(8, 5))
    plt.plot(log["epoch"], log["train_iou"], label="train online IoU")
    plt.plot(log["epoch"], log["val_iou"], label="validation IoU")
    plt.xlabel("epoch"); plt.ylabel("IoU"); plt.legend(); plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(log["epoch"], log["train_loss"], label="train loss")
    plt.plot(log["epoch"], log["val_loss"], label="validation loss")
    plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.show()
else:
    print("No training log yet:", log_path)


## 9. Diagnose the selected checkpoint before test inference


In [ ]:
BEST_CHECKPOINT = RUN_DIR / "best_by_val_iou.pth"
DIAGNOSIS_PATH = RUN_DIR / "checkpoint_diagnosis.json"

if BEST_CHECKPOINT.is_file() and HIERA_PATH.is_file():
    subprocess.check_call([
        sys.executable, str(PROJECT_DIR / "diagnose_checkpoint.py"),
        "--checkpoint", str(BEST_CHECKPOINT),
        "--hiera_path", str(HIERA_PATH),
        "--val_image_path", str(VAL_IMAGE),
        "--val_mask_path", str(VAL_MASK),
        "--val_dtm_path", str(VAL_DTM),
        "--output_json", str(DIAGNOSIS_PATH),
    ])
else:
    print("Diagnosis skipped; missing checkpoint or HIERA file:", BEST_CHECKPOINT, HIERA_PATH)


## 10. Test inference

The submitted `test.sh` loaded the **100% DTM** checkpoint but set `--dtm_availability 0.0`. The corrected matching condition is `1.0`. Use `0.0` only for an explicitly named no-DTM ablation output folder.


In [ ]:
TEST_COMMAND = [
    sys.executable, str(PROJECT_DIR / "test.py"),
    "--checkpoint", str(BEST_CHECKPOINT),
    "--hiera_path", str(HIERA_PATH),
    "--test_image_path", str(TEST_IMAGE),
    "--test_dtm_path", str(TEST_DTM),
    "--test_gt_path", str(TEST_MASK),
    "--save_path", str(PRED_DIR),
    "--dtm_availability", "1.0",  # matching 100% DTM condition
]
print(" ".join(map(str, TEST_COMMAND)))

if RUN_TESTING:
    subprocess.check_call(TEST_COMMAND)
else:
    print("Testing skipped. Set RUN_TESTING=True after a valid checkpoint exists.")


## 11. Evaluate fixed-threshold predictions

The corrected evaluator reports pooled IoU at the actual binary threshold. This is directly comparable to training/validation. The old evaluator's dynamic-threshold PySOD `mIoU` is a different metric and is now labelled separately.


In [ ]:
EVAL_COMMAND = [
    sys.executable, str(PROJECT_DIR / "eval.py"),
    "--dataset_name", "test_100pct_dtm",
    "--pred_path", str(PRED_DIR),
    "--gt_path", str(TEST_MASK),
]
print(" ".join(map(str, EVAL_COMMAND)))

if RUN_EVALUATION:
    subprocess.check_call(EVAL_COMMAND)
else:
    print("Evaluation skipped. Set RUN_EVALUATION=True after predictions exist.")


## Acceptance criteria for the next run

Do not judge success from train IoU alone. A defensible run should satisfy all of the following:

1. Pairing audit: matched count equals intended RGB count; no duplicate stems; no hidden drops.
2. Split audit: zero identical stems across train and validation; preferably split by source corridor/chainage, not random neighbouring tiles.
3. DTM audit: finite ratio, nodata handling, value range, and visual co-registration are plausible.
4. Checkpoint audit: no global all-background/all-foreground collapse; precision and recall are both non-zero.
5. Validation: report IoU/Dice/precision/recall at the fixed operating threshold plus the threshold sweep.
6. Ablation: compare RGB-only (`availability=0.0`) and RGB+DTM (`1.0`) using the same seed, split, optimiser, and stopping rule. Output folders and manifests must state the actual availability.
7. Repeatability: run at least three seeds before claiming a DTM gain; report mean and standard deviation.
